# 4N — Diffusion Convolutional Recurrent Neural Network (Weekly DCRNN)

## Objective

Train the neural architecture on **weekly state-level accident counts**. Its one-step-ahead output is a count for every state node; summing all node predictions produces the same national weekly target predicted by Stream I.

This notebook adapts the supplied hourly DCRNN design. It removes the California filter, 5 km grid, 128-node truncation, multi-hour horizon, severity channel, solar/hourly covariates and event-alert classification report. Instead it uses the complete set of state nodes created in Notebook 3N and a one-week horizon.

### Explicit compute limits

- GPU required; this notebook stops if CUDA is unavailable.
- Optuna: at most 12 trials, one trial at a time.
- Expanding validation: 2 folds.
- At most 12 epochs per fold, patience 3.
- Optuna wall-clock timeout: 45 minutes.
- Final refit: at most 10 epochs, using the median best fold epoch from the selected trial.

The time cap applies to Optuna search; the bounded final refit is additional. Reduce the constants in Section 0 before starting if you need a stricter allocation.

The evaluation protocol is the same as the incremental models: fitted weights are frozen for test evaluation, while actual newly observed weekly counts are allowed into the historical input window for the following one-step forecast.

## 0. Dependencies, GPU check, paths and hard compute limits

In [ ]:
from pathlib import Path
import os, json, warnings, gc, time, copy, random, sys
import numpy as np
import pandas as pd

SEED = 34
random.seed(SEED)
np.random.seed(SEED)

ON_KAGGLE = Path('/kaggle/working').exists()
PROJECT_ROOT = Path('/kaggle/working/us_accidents_weekly_forecasting') if ON_KAGGLE else Path('./us_accidents_weekly_forecasting')
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts'
MODEL_DIR = PROJECT_ROOT / 'models'
REPORT_DIR = PROJECT_ROOT / 'reports'
for _p in (PROJECT_ROOT, ARTIFACT_DIR, MODEL_DIR, REPORT_DIR):
    _p.mkdir(parents=True, exist_ok=True)

# Set this only when more than one prior notebook output is attached as a Kaggle input.
ARTIFACT_INPUT_ROOT = None  # e.g. Path('/kaggle/input/us-accidents-weekly-stage-2')

def resolve_artifact(filename, required=True):
    candidates = []
    if ARTIFACT_INPUT_ROOT is not None:
        p = Path(ARTIFACT_INPUT_ROOT) / filename
        if p.exists():
            candidates.append(p)
    p = ARTIFACT_DIR / filename
    if p.exists():
        candidates.append(p)
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        candidates.extend(kaggle_input.rglob(filename))
    # Deduplicate resolved paths and prefer the most recently modified attached artifact.
    unique = {str(p.resolve()): p for p in candidates}
    candidates = list(unique.values())
    if not candidates:
        if required:
            raise FileNotFoundError(
                f'Could not locate {filename}. Add the preceding notebook output as a Kaggle input, '
                f'or set ARTIFACT_INPUT_ROOT. Local output searched: {ARTIFACT_DIR}'
            )
        return None
    chosen = max(candidates, key=lambda p: p.stat().st_mtime)
    if len(candidates) > 1:
        print(f'Found {len(candidates)} copies of {filename}; using newest: {chosen}')
    return chosen

print('Project root:', PROJECT_ROOT.resolve())
print('Seed:', SEED)

import math, gc, json, time, copy, random
import matplotlib.pyplot as plt
import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING)
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

RANDOM_STATE = 34
MAX_OPTUNA_TRIALS = 12
MAX_OPTUNA_MINUTES = 45
N_CV_FOLDS = 2
CV_TEST_WEEKS = 8
MAX_EPOCHS_PER_FOLD = 12
EARLY_STOPPING_PATIENCE = 3
MAX_FINAL_REFIT_EPOCHS = 10
STUDY_NAME = 'us_accidents_weekly_dcrnn_state_graph'

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU is required for 4N. In Kaggle, set Accelerator = GPU before running.')
DEVICE = torch.device('cuda:0')
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
torch.cuda.manual_seed_all(RANDOM_STATE)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
try:
    torch.set_float32_matmul_precision('high')
except Exception:
    pass

print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('Optuna cap:', MAX_OPTUNA_TRIALS, 'trials /', MAX_OPTUNA_MINUTES, 'minutes')

## 1. Load the weekly graph tensors

`Y_COUNTS[t, n]` is the count for week index `t` and state node `n`. `C_RAW[t, n, :]` contains future-known calendar and state identity covariates. `week_id` is the sequential weekly group identifier.

In [ ]:
Y_COUNTS = np.load(resolve_artifact('dcrnn_weekly_state_counts.npy')).astype(np.float32)
C_RAW = np.load(resolve_artifact('dcrnn_future_covariates.npy')).astype(np.float32)
ADJ = np.load(resolve_artifact('dcrnn_adjacency.npy')).astype(np.float32)
WEEK_IDS = np.load(resolve_artifact('dcrnn_week_ids.npy')).astype(np.int32)
WEEK_DATES = np.load(resolve_artifact('dcrnn_week_dates.npy')).astype('datetime64[ns]')
DCRNN_META = json.loads(resolve_artifact('dcrnn_weekly_preprocessing_metadata.json').read_text())
SPLIT_META = json.loads(resolve_artifact('split_metadata.json').read_text())
STATES = DCRNN_META['state_nodes']
T, N = Y_COUNTS.shape
TRAIN_END_WEEK_ID = int(SPLIT_META['train_end_week_id'])
TRAIN_END_IDX = int(np.searchsorted(WEEK_IDS, TRAIN_END_WEEK_ID, side='right'))

if C_RAW.shape[:2] != (T, N) or ADJ.shape != (N, N):
    raise AssertionError('DCRNN tensors, graph and node mapping are not aligned.')
if len(STATES) != N or not np.isfinite(Y_COUNTS).all() or not np.isfinite(C_RAW).all():
    raise AssertionError('DCRNN tensor metadata validation failed.')
if TRAIN_END_IDX < 100 or T - TRAIN_END_IDX < 10:
    raise ValueError('Insufficient training/test weeks for DCRNN.')
print('T:', T, '| nodes:', N, '| train weeks:', TRAIN_END_IDX, '| test weeks:', T-TRAIN_END_IDX)
print('Covariate dimension:', C_RAW.shape[-1])

## 2. DCRNN graph support and model architecture

In [ ]:
def row_normalize(matrix):
    degree = matrix.sum(axis=1, keepdims=True)
    return np.divide(matrix, np.maximum(degree, 1e-12), out=np.zeros_like(matrix), where=degree > 0)


def sparse_support(matrix, device):
    rows, cols = np.nonzero(matrix)
    indices = torch.tensor(np.vstack([rows, cols]), dtype=torch.long, device=device)
    values = torch.tensor(matrix[rows, cols], dtype=torch.float32, device=device)
    return torch.sparse_coo_tensor(indices, values, size=matrix.shape, dtype=torch.float32, device=device).coalesce()


def make_supports(adjacency):
    forward = row_normalize(adjacency.astype(np.float32))
    reverse = row_normalize(adjacency.T.astype(np.float32))
    return sparse_support(forward, DEVICE), sparse_support(reverse, DEVICE)


class DiffusionConv(nn.Module):
    def __init__(self, in_channels, out_channels, supports, diffusion_steps):
        super().__init__()
        self.register_buffer('support_forward', supports[0], persistent=False)
        self.register_buffer('support_reverse', supports[1], persistent=False)
        self.diffusion_steps = int(diffusion_steps)
        n_terms = 1 + 2 * self.diffusion_steps
        self.linear = nn.Linear(in_channels * n_terms, out_channels)

    @staticmethod
    def propagate(support, x):
        # x: [batch, nodes, channels]
        batch, nodes, features = x.shape
        flat = x.permute(1, 0, 2).contiguous().reshape(nodes, batch * features)
        out = torch.sparse.mm(support, flat)
        return out.reshape(nodes, batch, features).permute(1, 0, 2).contiguous()

    def forward(self, x):
        features = [x]
        forward_x = x
        reverse_x = x
        for _ in range(self.diffusion_steps):
            forward_x = self.propagate(self.support_forward, forward_x)
            reverse_x = self.propagate(self.support_reverse, reverse_x)
            features.extend([forward_x, reverse_x])
        return self.linear(torch.cat(features, dim=-1))


class DCGRUCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, supports, diffusion_steps):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.gates = DiffusionConv(input_dim + hidden_dim, 2 * hidden_dim, supports, diffusion_steps)
        self.candidate = DiffusionConv(input_dim + hidden_dim, hidden_dim, supports, diffusion_steps)

    def forward(self, x, hidden):
        reset, update = torch.chunk(torch.sigmoid(self.gates(torch.cat([x, hidden], dim=-1))), 2, dim=-1)
        candidate = torch.tanh(self.candidate(torch.cat([x, reset * hidden], dim=-1)))
        return update * hidden + (1.0 - update) * candidate


class WeeklyDCRNN(nn.Module):
    def __init__(self, input_dim, covariate_dim, hidden_dim, num_layers, supports, diffusion_steps=1, dropout=0.1):
        super().__init__()
        self.hidden_dim = int(hidden_dim)
        self.num_layers = int(num_layers)
        self.dropout = float(dropout)
        self.encoder_cells = nn.ModuleList()
        self.decoder_cells = nn.ModuleList()
        for layer in range(self.num_layers):
            encoder_in = input_dim if layer == 0 else hidden_dim
            decoder_in = 1 + covariate_dim if layer == 0 else hidden_dim
            self.encoder_cells.append(DCGRUCell(encoder_in, hidden_dim, supports, diffusion_steps))
            self.decoder_cells.append(DCGRUCell(decoder_in, hidden_dim, supports, diffusion_steps))
        self.projection = nn.Linear(hidden_dim, 1)

    def _stack_step(self, x, states, cells):
        new_states = []
        layer_input = x
        for layer, cell in enumerate(cells):
            h_new = cell(layer_input, states[layer])
            new_states.append(h_new)
            layer_input = F.dropout(h_new, p=self.dropout, training=self.training) if layer < len(cells)-1 else h_new
        return new_states

    def forward(self, x_history, future_covariates):
        # x_history [B, history, nodes, input_dim]; future covariates [B, 1, nodes, cov_dim].
        batch, history, nodes, _ = x_history.shape
        states = [x_history.new_zeros((batch, nodes, self.hidden_dim)) for _ in range(self.num_layers)]
        for step in range(history):
            states = self._stack_step(x_history[:, step], states, self.encoder_cells)
        previous_count = x_history[:, -1, :, :1]
        decoder_input = torch.cat([previous_count, future_covariates[:, 0]], dim=-1)
        states = self._stack_step(decoder_input, states, self.decoder_cells)
        prediction = self.projection(states[-1])
        return prediction.unsqueeze(1)

print('DCRNN architecture defined.')

## 3. Target normalization and lazy sliding-window datasets

In [ ]:
def fit_log_target_scaler(counts_until_index):
    log_values = np.log1p(np.maximum(counts_until_index, 0.0)).astype(np.float64)
    mean = float(log_values.mean())
    std = float(max(log_values.std(), 1e-6))
    return mean, std


def make_normalized_tensors(target_mean, target_std):
    log_y = np.log1p(np.maximum(Y_COUNTS, 0.0))
    y_norm = ((log_y - target_mean) / target_std).astype(np.float32)
    X_full = np.concatenate([y_norm[:, :, None], C_RAW], axis=-1).astype(np.float32)
    Y_full = y_norm[:, :, None].astype(np.float32)
    return X_full, Y_full


def inverse_counts(y_norm, target_mean, target_std):
    log_counts = np.asarray(y_norm, dtype=np.float64) * target_std + target_mean
    return np.maximum(np.expm1(np.clip(log_counts, -20.0, 20.0)), 0.0).astype(np.float32)


class WeeklyWindowDataset(Dataset):
    def __init__(self, X, Y, C, target_starts, history):
        self.X, self.Y, self.C = X, Y, C
        self.starts = np.asarray(target_starts, dtype=np.int64)
        self.history = int(history)
    def __len__(self):
        return len(self.starts)
    def __getitem__(self, index):
        start = int(self.starts[index])
        x = np.array(self.X[start-self.history:start], dtype=np.float32, copy=True)
        y = np.array(self.Y[start:start+1], dtype=np.float32, copy=True)
        c = np.array(self.C[start:start+1], dtype=np.float32, copy=True)
        return torch.from_numpy(x), torch.from_numpy(y), torch.from_numpy(c), torch.tensor(start, dtype=torch.long)


def make_loader(X, Y, C, starts, history, batch_size, shuffle):
    ds = WeeklyWindowDataset(X, Y, C, starts, history)
    generator = torch.Generator().manual_seed(RANDOM_STATE)
    return DataLoader(ds, batch_size=int(batch_size), shuffle=shuffle, num_workers=0,
                      pin_memory=True, drop_last=False, generator=generator)


def score_national_mae(model, loader, target_mean, target_std):
    model.eval()
    errors = []
    with torch.no_grad():
        for x, y, cov, starts in loader:
            x = x.to(DEVICE, non_blocking=True)
            cov = cov.to(DEVICE, non_blocking=True)
            pred_norm = model(x, cov)[:, 0, :, 0].detach().cpu().numpy()
            pred_nodes = inverse_counts(pred_norm, target_mean, target_std)
            pred_national = pred_nodes.sum(axis=1)
            true_national = Y_COUNTS[starts.numpy()].sum(axis=1)
            errors.extend(np.abs(pred_national - true_national).tolist())
    return float(np.mean(errors)) if errors else float('inf')


def seed_everything(seed=RANDOM_STATE):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

## 4. Expanding-window validation and bounded Optuna search

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

HISTORY_CHOICES = [13, 26, 52]


def get_folds(history):
    starts = np.arange(history, TRAIN_END_IDX, dtype=np.int64)
    splitter = TimeSeriesSplit(n_splits=N_CV_FOLDS, test_size=CV_TEST_WEEKS)
    return [(starts[tr], starts[va]) for tr, va in splitter.split(starts)]


def train_fold(train_starts, val_starts, history, params, fold_number):
    seed_everything(RANDOM_STATE)
    val_start = int(val_starts[0])
    target_mean, target_std = fit_log_target_scaler(Y_COUNTS[:val_start])
    X_full, Y_full = make_normalized_tensors(target_mean, target_std)
    supports = make_supports(ADJ)
    model = WeeklyDCRNN(
        input_dim=X_full.shape[-1], covariate_dim=C_RAW.shape[-1],
        hidden_dim=params['hidden_dim'], num_layers=params['num_layers'],
        supports=supports, diffusion_steps=params['diffusion_steps'], dropout=params['dropout'],
    ).to(DEVICE)
    train_loader = make_loader(X_full, Y_full, C_RAW, train_starts, history, params['batch_size'], True)
    val_loader = make_loader(X_full, Y_full, C_RAW, val_starts, history, params['batch_size'], False)
    optimizer = torch.optim.AdamW(model.parameters(), lr=params['learning_rate'], weight_decay=params['weight_decay'])
    best_score = float('inf'); best_epoch = 1; best_state = None; wait = 0
    for epoch in range(1, MAX_EPOCHS_PER_FOLD + 1):
        model.train(); losses = []
        for x, y, cov, starts in train_loader:
            x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True); cov = cov.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            pred = model(x, cov)
            loss = F.l1_loss(pred, y)
            if not torch.isfinite(loss):
                raise FloatingPointError('Non-finite DCRNN training loss.')
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            losses.append(float(loss.detach().item()))
        val_mae = score_national_mae(model, val_loader, target_mean, target_std)
        if val_mae < best_score:
            best_score = val_mae; best_epoch = epoch; wait = 0
            best_state = {k: v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else:
            wait += 1
        if wait >= EARLY_STOPPING_PATIENCE:
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    del model, optimizer, train_loader, val_loader, X_full, Y_full
    gc.collect(); torch.cuda.empty_cache()
    return best_score, best_epoch


def suggest_dcrnn_params(trial):
    return {
        'history_weeks': trial.suggest_categorical('history_weeks', HISTORY_CHOICES),
        'hidden_dim': trial.suggest_categorical('hidden_dim', [8, 16, 32]),
        'num_layers': trial.suggest_int('num_layers', 1, 2),
        'diffusion_steps': trial.suggest_int('diffusion_steps', 1, 2),
        'dropout': trial.suggest_float('dropout', 0.0, 0.25),
        'learning_rate': trial.suggest_float('learning_rate', 1e-4, 5e-3, log=True),
        'weight_decay': trial.suggest_float('weight_decay', 1e-7, 1e-3, log=True),
        'batch_size': trial.suggest_categorical('batch_size', [8, 16, 32]),
    }


def objective(trial):
    params = suggest_dcrnn_params(trial)
    folds = get_folds(params['history_weeks'])
    fold_scores, fold_epochs = [], []
    for fold_id, (train_starts, val_starts) in enumerate(folds):
        score, best_epoch = train_fold(train_starts, val_starts, params['history_weeks'], params, fold_id)
        fold_scores.append(score); fold_epochs.append(best_epoch)
        trial.report(float(np.mean(fold_scores)), step=fold_id)
        if trial.should_prune():
            raise optuna.TrialPruned()
    trial.set_user_attr('fold_mae', fold_scores)
    trial.set_user_attr('fold_best_epochs', fold_epochs)
    return float(np.mean(fold_scores))

STUDY_DB = MODEL_DIR / 'weekly_dcrnn_optuna.sqlite3'
study = optuna.create_study(
    study_name=STUDY_NAME, direction='minimize',
    sampler=TPESampler(seed=RANDOM_STATE, n_startup_trials=4),
    pruner=MedianPruner(n_startup_trials=4, n_warmup_steps=1),
    storage=f'sqlite:///{STUDY_DB}', load_if_exists=True,
)
remaining = max(0, MAX_OPTUNA_TRIALS - len(study.trials))
print('Existing trials:', len(study.trials), '| remaining trial allowance:', remaining)
if remaining:
    study.optimize(objective, n_trials=remaining, timeout=MAX_OPTUNA_MINUTES * 60,
                   n_jobs=1, gc_after_trial=True, show_progress_bar=True)
completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
if not completed:
    raise RuntimeError('No DCRNN trial completed. Check CUDA memory and trial logs.')
print('Best validation MAE:', study.best_value)
print('Best parameters:', study.best_params)
study.trials_dataframe().to_csv(REPORT_DIR / 'dcrnn_optuna_trials.csv', index=False)

## 5. Refit the selected DCRNN on the complete training period

In [ ]:
best_params = dict(study.best_params)
history = int(best_params.pop('history_weeks'))
fold_epochs = study.best_trial.user_attrs.get('fold_best_epochs', [MAX_FINAL_REFIT_EPOCHS])
final_epochs = int(np.clip(np.median(fold_epochs), 1, MAX_FINAL_REFIT_EPOCHS))
target_mean, target_std = fit_log_target_scaler(Y_COUNTS[:TRAIN_END_IDX])
X_full, Y_full = make_normalized_tensors(target_mean, target_std)
supports = make_supports(ADJ)
final_model = WeeklyDCRNN(
    input_dim=X_full.shape[-1], covariate_dim=C_RAW.shape[-1],
    hidden_dim=best_params['hidden_dim'], num_layers=best_params['num_layers'],
    supports=supports, diffusion_steps=best_params['diffusion_steps'], dropout=best_params['dropout'],
).to(DEVICE)
train_starts = np.arange(history, TRAIN_END_IDX, dtype=np.int64)
train_loader = make_loader(X_full, Y_full, C_RAW, train_starts, history, best_params['batch_size'], True)
optimizer = torch.optim.AdamW(final_model.parameters(), lr=best_params['learning_rate'], weight_decay=best_params['weight_decay'])
for epoch in range(1, final_epochs + 1):
    final_model.train(); losses = []
    for x, y, cov, starts in train_loader:
        x=x.to(DEVICE, non_blocking=True); y=y.to(DEVICE, non_blocking=True); cov=cov.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        pred=final_model(x,cov); loss=F.l1_loss(pred,y)
        loss.backward(); torch.nn.utils.clip_grad_norm_(final_model.parameters(),5.0); optimizer.step()
        losses.append(float(loss.detach().item()))
    print(f'Refit epoch {epoch}/{final_epochs} — normalized log-count MAE: {np.mean(losses):.5f}')

# Frozen model, rolling-origin one-step forecasts. Actual observed test weeks enter later input windows.
test_starts = np.arange(TRAIN_END_IDX, T, dtype=np.int64)
test_loader = make_loader(X_full, Y_full, C_RAW, test_starts, history, best_params['batch_size'], False)
final_model.eval()
week_pred_nodes=[]; week_true_nodes=[]; week_ids=[]
with torch.no_grad():
    for x, y, cov, starts in test_loader:
        x=x.to(DEVICE, non_blocking=True); cov=cov.to(DEVICE, non_blocking=True)
        pred_norm=final_model(x,cov)[:,0,:,0].detach().cpu().numpy()
        pred_nodes=inverse_counts(pred_norm,target_mean,target_std)
        true_nodes=Y_COUNTS[starts.numpy()]
        week_pred_nodes.append(pred_nodes); week_true_nodes.append(true_nodes); week_ids.extend(starts.numpy().tolist())

pred_nodes=np.concatenate(week_pred_nodes,axis=0)
true_nodes=np.concatenate(week_true_nodes,axis=0)
actual_national=true_nodes.sum(axis=1)
pred_national=pred_nodes.sum(axis=1)

test_rows=pd.DataFrame({
    'target_week_id': WEEK_IDS[np.asarray(week_ids)],
    'target_week_start': pd.to_datetime(WEEK_DATES[np.asarray(week_ids)]),
    'actual_count': actual_national.astype(float),
    'predicted_count': pred_national.astype(float),
    'model': 'DCRNN',
})
test_rows.to_csv(REPORT_DIR / 'dcrnn_test_predictions.csv',index=False)

# Retain per-state predictions for spatial diagnostics; all nodes remain represented.
state_rows=[]
for r, t_idx in enumerate(week_ids):
    for node_id, state in enumerate(STATES):
        state_rows.append({
            'target_week_id': int(WEEK_IDS[t_idx]), 'target_week_start': pd.Timestamp(WEEK_DATES[t_idx]),
            'state': state, 'actual_count': float(true_nodes[r,node_id]),
            'predicted_count': float(pred_nodes[r,node_id]),
        })
pd.DataFrame(state_rows).to_csv(REPORT_DIR / 'dcrnn_state_test_predictions.csv',index=False)

checkpoint_path=MODEL_DIR / 'weekly_dcrnn_state_graph.pt'
torch.save({
    'model_state_dict': {k:v.detach().cpu() for k,v in final_model.state_dict().items()},
    'model_config': {
        'input_dim': int(X_full.shape[-1]), 'covariate_dim': int(C_RAW.shape[-1]),
        'hidden_dim': int(best_params['hidden_dim']), 'num_layers': int(best_params['num_layers']),
        'diffusion_steps': int(best_params['diffusion_steps']), 'dropout': float(best_params['dropout']),
        'history_weeks': history, 'forecast_horizon_weeks': 1,
    },
    'best_params': best_params,
    'target_mean': target_mean, 'target_std': target_std,
    'states': STATES, 'adjacency': ADJ,
    'week_definition': SPLIT_META['week_definition'],
    'random_state': RANDOM_STATE,
}, checkpoint_path)

metadata={
    'model':'weekly state-graph DCRNN', 'best_cv_national_mae':float(study.best_value),
    'best_params':best_params, 'history_weeks':history, 'final_epochs':final_epochs,
    'n_trials_total':len(study.trials), 'completed_trials':len(completed),
    'optuna_timeout_minutes':MAX_OPTUNA_MINUTES, 'n_cv_folds':N_CV_FOLDS,
    'max_epochs_per_fold':MAX_EPOCHS_PER_FOLD, 'early_stopping_patience':EARLY_STOPPING_PATIENCE,
    'device':str(DEVICE), 'gpu_name':torch.cuda.get_device_name(0),
    'n_nodes':N, 'state_nodes':STATES, 'adjusted_r2_feature_count':int(C_RAW.shape[-1]+1),
    'prediction_file':str(REPORT_DIR/'dcrnn_test_predictions.csv'), 'checkpoint':str(checkpoint_path),
    'evaluation_protocol':SPLIT_META['evaluation_protocol'],
}
(REPORT_DIR/'dcrnn_metadata.json').write_text(json.dumps(metadata,indent=2,default=str))
print('Saved DCRNN checkpoint:',checkpoint_path)
print('Saved national predictions:',REPORT_DIR/'dcrnn_test_predictions.csv')

## 6. Quick inspection (final metrics are consolidated in Notebook 6)

In [ ]:
display(test_rows.head(10))
fig, ax = plt.subplots(figsize=(14,4))
ax.plot(test_rows.target_week_start,test_rows.actual_count,label='Actual',color='blue')
ax.plot(test_rows.target_week_start,test_rows.predicted_count,label='DCRNN',alpha=0.85)
ax.set_title('DCRNN — one-week-ahead national accident-count forecasts')
ax.set_xlabel('Target week'); ax.set_ylabel('Accident reports')
ax.legend(); ax.grid(alpha=0.25); plt.tight_layout(); plt.show()